##### Import libraries

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import json
import unicodedata
import spacy
from spacy.matcher import PhraseMatcher
from spacy.tokens import Token

pd.set_option("display.max_columns", None)   # no column truncation
pd.set_option("display.max_rows", None)      # no row truncation
pd.set_option("display.width", None)         # no line-wrapping
pd.set_option("display.max_colwidth", None)

### Configuration and Functions used in the notebook

##### ACE Terms dictionary

Moved to a json file. Load it at the beginning of the annotation pipeline.

##### Regex configuration

In [2]:
# All these cues can be improved according to needs.

# Negation cues. 
NEG_CUES = re.compile(
    r"\b("
    r"niega(?:n)?|"
    r"sin|"
    r"descarta(?:n)?|"
    r"no\s+(?:refiere|presenta|consta|evidencia|hay|ha|han|tuvo|tiene|presentó|refirió)"
    r")\b",
    re.I
)

# Temporal cues
CHILD_CUES = re.compile(
    r"\b("
    # Life stages
    r"(?:en|desde|durante)\s+(?:la|su)\s+"
    r"(?:infancia|niñez|adolescencia|infancia\s+temprana|infancia\s+precoz)|"

    # When he/she was...
    r"cuando\s+era\s+(?:niñ[oa]|menor|adolescente|pequeñ[oa])|"

    # As a child / when young
    r"de\s+(?:niñ[oa]|pequeñ[oa])|"
    r"desde\s+(?:niñ[oa]|pequeñ[oa])|"
    r"siendo\s+(?:niñ[oa]|menor|adolescente|pequeñ[oa])|"

    # Explicit minor
    r"menor\s+de\s+edad|"

    # Explicit age 0-17
    r"(?:a\s+los|con)\s+(?:[0-9]|1[0-7])\s+a[nñ]os|"

    # Adolescence variants
    r"en\s+(?:su\s+)?adolescencia|"
    r"durante\s+la\s+adolescencia|"

    # Infancy variants without article rigidity
    r"en\s+(?:su\s+)?infancia|"
    r"durante\s+(?:la|su)\s+niñez|"
    r"desde\s+(?:la|su)\s+infancia"
    r")\b",
    re.I
)

RECENT_CUES = re.compile(
    r"\b("
    # relative durations
    r"(?:desde\s+hace|hace)\s+\d+\s+(?:a[nñ]os|mes(?:es)?|semanas|d[ií]as|horas)|"
    # "last N days/weeks/months/years"
    r"en\s+los\s+[uú]ltimos\s+\d+\s+(?:d[ií]as|semanas|mes(?:es)?|a[nñ]os)|"
    # adverbs
    r"recientemente|actualmente|en\s+la\s+actualidad|"
    # "last week/month/year", "this week/month/year"
    r"(?:la|este|esta)\s+(?:semana|mes|a[nñ]o)"
    r")\b",
    re.I
)

# Perpretator cues
PERPETRATOR_LINK_CUES = re.compile(
    r"\b("
    r"por\s+parte\s+de(?:l|la|los|las)?|"
    r"a\s+manos\s+de(?:l|la|los|las)?|"
    r"ejercid[oa]\s+por|"
    r"causad[oa]\s+por|"
    r"cometid[oa]\s+por|"
    r"perpetrad[oa]\s+por"
    r")\b",
    re.I
)

PERPETRATOR_CUES = re.compile(
    r"\b("
    r"padre\s+de\s+su\s+hij[oa]|"
    r"madre\s+de\s+su\s+hij[oa]|"
    r"padre|madre|"
    r"herman[oa]|"
    r"t[ií][oa]|"
    r"abuel[oa]|"
    r"prim[oa]|"
    r"profesor|profesora|"
    r"maestro|maestra"
    r")\b",
    re.I
)

In [3]:
# Possible separations between clauses in a sentence
CLAUSE_BREAKS = re.compile(r"[,:;]|\b(?:y|pero|aunque|sin embargo)\b", re.I)

##### Functions

In [4]:
# Functions to extract spans to the L or to the R of a central word.
def cut_left_at_break(text):
    matches = list(CLAUSE_BREAKS.finditer(text))
    return text[matches[-1].end():] if matches else text

def cut_right_at_break(text):
    m = CLAUSE_BREAKS.search(text)
    return text[:m.start()] if m else text

In [ ]:
# Funtion to extract attribute values related to temporal cues, in childhood and recent times.
# Windows can be configured to the L and to the R.
def temporal_flags_local(span, left_window=40, right_window=40):
    text = span.doc.text

    L = max(span.start_char - left_window, 0)
    R = min(span.end_char + right_window, len(text))

    ctx_left = text[L:span.start_char]
    ctx_right = text[span.end_char:R]

    # Keep only nearest local fragment
    ctx_left = cut_left_at_break(ctx_left)
    ctx_right = cut_right_at_break(ctx_right)

    child_match = CHILD_CUES.search(ctx_left) or CHILD_CUES.search(ctx_right)
    recent_match = RECENT_CUES.search(ctx_left) or RECENT_CUES.search(ctx_right)

    childhood = bool(child_match)
    recent = bool(recent_match)

    childhood_term = child_match.group(0) if child_match else None
    recent_term = recent_match.group(0) if recent_match else None

    return childhood, childhood_term, recent, recent_term

In [6]:
# Funtion to extract attribute values related to perpetrator cues.
# Windows can be configured to the L and to the R.
def perpetrator_flags_local(span, left_window=80, right_window=30):
    text = span.doc.text

    L = max(span.start_char - left_window, 0)
    R = min(span.end_char + right_window, len(text))

    ctx_left = text[L:span.start_char]
    ctx_right = text[span.end_char:R]

    # Restrict to nearest clause
    ctx_left = cut_left_at_break(ctx_left)
    # print("left: ", ctx_left)
    ctx_right = cut_right_at_break(ctx_right)
    # print("right: ", ctx_right)

    # Look into left and right context
    left_term = PERPETRATOR_CUES.search(ctx_left)
    left_link = PERPETRATOR_LINK_CUES.search(ctx_left)

    right_term = PERPETRATOR_CUES.search(ctx_right)
    right_link = PERPETRATOR_LINK_CUES.search(ctx_right)

    # Strong rule: require both a perpetrator term and a link cue 
    # (introduced to avoid false assignations for perpetrator)
    if left_term and left_link:
        return True, left_term.group(0)

    if right_term and right_link:
        return True, right_term.group(0)

    return False, None

In [7]:
def annotate_ace(text: str, nlp, matcher):
    doc = nlp(text)
    matches = matcher(doc)

    ann = []
    for match_id, start, end in matches:
        span = doc[start:end]

        # Sentence-bounded context (left/right of the matched span)
        sent = span.sent.text
        rel_start = span.start_char - span.sent.start_char
        rel_end   = span.end_char   - span.sent.start_char
        sent_left  = sent[:rel_start]
        sent_right = sent[rel_end:]

        # Negation cues: check both sides (often left is enough, but this is safer)
        negated = bool(NEG_CUES.search(sent_left) or NEG_CUES.search(sent_right))

        # Temporal cues split into childhood vs recent
        childhood, childhood_term, recent, recent_term = temporal_flags_local(
    span, left_window=40, right_window=40)
        
        # Perpetrator cues
        perpetrator_cue, perpetrator_term = perpetrator_flags_local(span, left_window=50, right_window=50)
        # Create the annotations
        ann.append({
            "concept_id": nlp.vocab.strings[match_id],  # category or could be SNOMED code (to be refined/changed according to feedback)
            "span": span.text,
            "start_char": span.start_char,
            "end_char": span.end_char,
            "negated": negated,
            "childhood_cue": childhood,
            "childhood_term": childhood_term,
            "recent_cue": recent,
            "recent_term": recent_term,
            "perpetrator_cue": perpetrator_cue,
            "perpetrator_term": perpetrator_term,
        })

    # Deduplicate exact spans (keep first)
    ann = sorted(ann, key=lambda x: (x["start_char"], -x["end_char"]))
    out, seen = [], set()
    for a in ann:
        key = (a["concept_id"], a["start_char"], a["end_char"])
        if key not in seen:
            out.append(a)
            seen.add(key)

    return out

In [8]:
# Function that takes out accents, maintains "ñ"
def strip_accents_keep_enye(text):
    result = []
    for char in text:
        if char in ("ñ", "Ñ"):
            result.append(char)
        else:
            decomposed = unicodedata.normalize("NFD", char)
            base = "".join(c for c in decomposed if unicodedata.category(c) != "Mn")
            result.append(base)
    return "".join(result)

In [10]:
def annotate_notes_df(df_in, note_id_column_orig, text_column_orig, cols_order, nlp, matcher,
                      note_id_column_dest="ehr_id",  
                      text_column_dest="ehr_history"):
    """Arguments:
    - df_in: dataframe with EHRs to annotate.
    - note_id_column_orig: name of the column that contains the EHR id in df_in.
    - text_column_orig: name of the column that contains the medical history to annotate in df_in.
    - note_id_column_dest: name of the column that contains the EHR id in the annotated df.
    - text_column_dest: name of the column that contains the medical history in the annotated df.
    - cols_order: desired column order for the resulting annotation dataframe.
    nlp: nlp object in the pipeline.
    matcher: matcher object in the pipeline.
    """
    all_ann = []

    for _, row in df_in.iterrows():
        note_text = row[text_column_orig]

        # Skip missing / empty notes
        if pd.isna(note_text) or not str(note_text).strip():
            continue

        ann = annotate_ace(str(note_text),nlp, matcher)

        # If no annotations, skip
        if not ann:
            continue

        ann_df = pd.DataFrame(ann)
        ann_df[note_id_column_dest] = row[note_id_column_orig]
        ann_df[text_column_dest] = note_text
        

        # Keep only desired columns in desired order
        ann_df = ann_df[cols_order]
        all_ann.append(ann_df)

    if all_ann:
        return pd.concat(all_ann, ignore_index=True)
    else:
        return pd.DataFrame(columns=cols_order)

In [11]:
def build_ace_matcher(nlp, ace_terms):
    matcher = PhraseMatcher(nlp.vocab, attr="LOWER")

    for concept_id, terms in ace_terms.items():
        pattern_texts = set()

        if not isinstance(terms, list):
            continue

        for term in terms:
            if not isinstance(term, str):
                continue

            term = term.strip()
            if not term:
                continue

            pattern_texts.add(term.lower())
            pattern_texts.add(strip_accents_keep_enye(term).lower())

        if pattern_texts:
            patterns = [nlp.make_doc(t) for t in pattern_texts]
            matcher.add(concept_id, patterns)

    return matcher

### Load abusos file

In [12]:
# Recover the path
directory_path = os.getcwd()
# root_path = os.path.dirname(directory_path)
print(directory_path)
# print(root_path)

c:\Users\adeli\Documents 4-Q2\Practicas-ACE\anotacionACE\ACE Project


In [13]:
# Patients in the file adversidad_precoz.xlsx (confirmed cases)
ace_cases_filepath = os.path.join(directory_path, 'data\\', "ace_confirmed_cases_eng.xlsx")
print(ace_cases_filepath)
ace_cases_df = pd.read_excel(ace_cases_filepath, na_values=["#NULL!"])

c:\Users\adeli\Documents 4-Q2\Practicas-ACE\anotacionACE\ACE Project\data\ace_confirmed_cases_eng.xlsx


In [14]:
len(ace_cases_df)

89

In [ ]:
ace_cases_df.head()

### Annotation pipeline

In [15]:
# Load the dictionary of ACE terms
with open("ace_terms_dictionary.json", "r", encoding="utf-8") as f:
    ace_terms = json.load(f)

In [ ]:
# Load the spanish medium model, build the matcher.
nlp = spacy.load("es_core_news_md", disable=["ner", "parser", "tagger"])
nlp.add_pipe("sentencizer")
matcher = build_ace_matcher(nlp, ace_terms)

In [17]:
# Choose the id column in the ACE cases dataframe to use and the text column to annotate
case_history_col = "Psychiatric_History"
case_id_col = "Anonymized_Temporal"

In [22]:
# Choose the names of the id and history columns in the resulting dataframe
id_col_name= "ehr_id"
history_col_name = "ehr_history"

# Choose the order of the columns in the resulting dataframe
cols_order = [
    id_col_name,
    "concept_id",
    "span",
    "negated",
    "childhood_cue",
    "childhood_term",
    "recent_cue",
    "recent_term",
    "perpetrator_cue",
    "perpetrator_term",
    history_col_name,
    "start_char",
    "end_char",
]

In [23]:
annotations_df = annotate_notes_df(
    ace_cases_df,
    text_column_orig= case_history_col,
    note_id_column_orig= case_id_col, 
    cols_order= cols_order, 
    nlp=nlp,
    matcher=matcher,
    note_id_column_dest=id_col_name,  
    text_column_dest=history_col_name
)

In [25]:
annotations_df.drop(columns=history_col_name).head(10)

,ehr_id,concept_id,span,negated,childhood_cue,childhood_term,recent_cue,recent_term,perpetrator_cue,perpetrator_term,start_char,end_char
0,216535,ACE_GENERAL_TRAUMA,victima,False,False,None,False,None,True,padre de su hijo,148,155
1,216535,ACE_PHYSICAL_ABUSE,malos tratos,False,False,None,False,None,True,padre de su hijo,159,171
2,216535,ACE_SEXUAL_ABUSE,abusos sexuales,False,True,en la infancia,False,None,False,None,206,221
3,216535,ACE_PARENT_SUBSTANCE,Madre alcoholica,False,False,None,False,None,False,None,867,883
4,216535,ACE_UNSPECIFIED_NEGLECT,negligencia,False,False,None,False,None,False,None,1102,1113
5,232165,ACE_GENERAL_TRAUMA,maltrato,False,True,en la infancia,False,None,False,None,1894,1902
6,262678,ACE_GENERAL_TRAUMA,maltrato,False,False,None,False,None,False,None,153,161
7,262678,ACE_PARENT_DIVORCE_SEPARATION,Padres separados,False,True,con 6 años,False,None,False,None,508,524
8,269487,ACE_PARENT_DIVORCE_SEPARATION,Padres separados,False,True,durante su infancia,False,None,False,None,423,439
9,269487,ACE_GENERAL_TRAUMA,maltrato,False,True,de niña,False,None,True,padre,728,736


In [ ]:
# Examine 271022